## Homework: Multilingual Embedding-based Machine Translation (7 points)

**In this homework** **<font color='red'>YOU</font>** will make machine translation system without using parallel corpora, alignment, attention, 100500 depth super-cool recurrent neural network and all that kind superstuff.

But even without parallel corpora this system can be good enough (hopefully). 

For our system we choose two kindred Slavic languages: Ukrainian and Russian. 

### Feel the difference!

(_синій кіт_ vs. _синій кит_)

![blue_cat_blue_whale.png](https://github.com/yandexdataschool/nlp_course/raw/master/resources/blue_cat_blue_whale.png)

### Fragment of the Swadesh list for some slavic languages

The Swadesh list is a lexicostatistical stuff. It's named after American linguist Morris Swadesh and contains basic lexis. This list are used to define subgroupings of languages, its relatedness.

So we can see some kind of word invariance for different Slavic languages.


| Russian         | Belorussian              | Ukrainian               | Polish             | Czech                         | Bulgarian            |
|-----------------|--------------------------|-------------------------|--------------------|-------------------------------|-----------------------|
| женщина         | жанчына, кабета, баба    | жінка                   | kobieta            | žena                          | жена                  |
| мужчина         | мужчына                  | чоловік, мужчина        | mężczyzna          | muž                           | мъж                   |
| человек         | чалавек                  | людина, чоловік         | człowiek           | člověk                        | човек                 |
| ребёнок, дитя   | дзіця, дзіцёнак, немаўля | дитина, дитя            | dziecko            | dítě                          | дете                  |
| жена            | жонка                    | дружина, жінка          | żona               | žena, manželka, choť          | съпруга, жена         |
| муж             | муж, гаспадар            | чоловiк, муж            | mąż                | muž, manžel, choť             | съпруг, мъж           |
| мать, мама      | маці, матка              | мати, матір, неня, мама | matka              | matka, máma, 'стар.' mateř    | майка                 |
| отец, тятя      | бацька, тата             | батько, тато, татусь    | ojciec             | otec                          | баща, татко           |
| много           | шмат, багата             | багато                  | wiele              | mnoho, hodně                  | много                 |
| несколько       | некалькі, колькі         | декілька, кілька        | kilka              | několik, pár, trocha          | няколко               |
| другой, иной    | іншы                     | інший                   | inny               | druhý, jiný                   | друг                  |
| зверь, животное | жывёла, звер, істота     | тварина, звір           | zwierzę            | zvíře                         | животно               |
| рыба            | рыба                     | риба                    | ryba               | ryba                          | риба                  |
| птица           | птушка                   | птах, птиця             | ptak               | pták                          | птица                 |
| собака, пёс     | сабака                   | собака, пес             | pies               | pes                           | куче, пес             |
| вошь            | вош                      | воша                    | wesz               | veš                           | въшка                 |
| змея, гад       | змяя                     | змія, гад               | wąż                | had                           | змия                  |
| червь, червяк   | чарвяк                   | хробак, черв'як         | robak              | červ                          | червей                |
| дерево          | дрэва                    | дерево                  | drzewo             | strom, dřevo                  | дърво                 |
| лес             | лес                      | ліс                     | las                | les                           | гора, лес             |
| палка           | кій, палка               | палиця                  | patyk, pręt, pałka | hůl, klacek, prut, kůl, pálka | палка, пръчка, бастун |

But the context distribution of these languages demonstrates even more invariance. And we can use this fact for our for our purposes.

## Data

In [9]:
import gensim
import numpy as np
from gensim.models import KeyedVectors

Download embeddings here:
* [cc.uk.300.vec.zip](https://yadi.sk/d/9CAeNsJiInoyUA)
* [cc.ru.300.vec.zip](https://yadi.sk/d/3yG0-M4M8fypeQ)

Load embeddings for ukrainian and russian.

In [12]:
uk_emb = KeyedVectors.load_word2vec_format("cc.uk.300.vec")

In [13]:
ru_emb = KeyedVectors.load_word2vec_format("cc.ru.300.vec")

In [14]:
ru_emb.most_similar([ru_emb["август"]], topn=10)

[('август', 0.9999999403953552),
 ('июль', 0.9383153915405273),
 ('сентябрь', 0.9240029454231262),
 ('июнь', 0.9222574830055237),
 ('октябрь', 0.9095540046691895),
 ('ноябрь', 0.8930034637451172),
 ('апрель', 0.8729087114334106),
 ('декабрь', 0.8652557134628296),
 ('март', 0.8545796275138855),
 ('февраль', 0.8401415944099426)]

In [15]:
uk_emb.most_similar([uk_emb["серпень"]])

[('серпень', 1.000000238418579),
 ('липень', 0.9096441268920898),
 ('вересень', 0.9016969203948975),
 ('червень', 0.8992519974708557),
 ('жовтень', 0.8810409307479858),
 ('листопад', 0.8787633180618286),
 ('квітень', 0.8592804074287415),
 ('грудень', 0.8586863279342651),
 ('травень', 0.8408111333847046),
 ('лютий', 0.8256431818008423)]

In [16]:
ru_emb.most_similar([uk_emb["серпень"]])

[('Недопустимость', 0.24435284733772278),
 ('конструктивность', 0.23293080925941467),
 ('офор', 0.23256801068782806),
 ('deteydlya', 0.23031717538833618),
 ('пресечении', 0.22632379829883575),
 ('одностороннего', 0.22608883678913116),
 ('подход', 0.2230587750673294),
 ('иболее', 0.22003719210624695),
 ('2015Александр', 0.21872766315937042),
 ('конструктивен', 0.21796567738056183)]

Load small dictionaries for correspoinding words pairs as trainset and testset.

In [18]:
def load_word_pairs(filename):
    uk_ru_pairs = []
    uk_vectors = []
    ru_vectors = []
    with open(filename, "r") as inpf:
        for line in inpf:
            uk, ru = line.rstrip().split("\t")
            if uk not in uk_emb or ru not in ru_emb:
                continue
            uk_ru_pairs.append((uk, ru))
            uk_vectors.append(uk_emb[uk])
            ru_vectors.append(ru_emb[ru])
    return uk_ru_pairs, np.array(uk_vectors), np.array(ru_vectors)

In [19]:
uk_ru_train, X_train, Y_train = load_word_pairs("ukr_rus.train.txt")

In [20]:
uk_ru_test, X_test, Y_test = load_word_pairs("ukr_rus.test.txt")

In [21]:
uk_ru_train[:10]

[('iснує', 'существует'),
 ('абияк', 'как-нибудь'),
 ('або', 'или'),
 ('або', 'ли'),
 ('абсолютно', 'совершенно'),
 ('абсолютно', 'совсем'),
 ('автомобіль', 'автомобиль'),
 ('автомобіль', 'вагон'),
 ('агов', 'эй'),
 ('аґрус', 'крыжовник')]

In [22]:
uk_ru_test[:10]

[('або', 'либо'),
 ('активний', 'активный'),
 ('актор', 'актер'),
 ('але', 'ж'),
 ('асамблея', 'собрание'),
 ('бабуся', 'бабушка'),
 ('багажник', 'ствол'),
 ('бажати', 'желать'),
 ('башта', 'башня'),
 ('бізнес', 'бизнес')]

In [23]:
X_train[:10]

array([[ 0.1638,  0.0508,  0.0287, ..., -0.0144,  0.0071,  0.1084],
       [ 0.1697, -0.0319,  0.0244, ..., -0.005 , -0.023 , -0.0275],
       [ 0.0818, -0.0011,  0.0303, ..., -0.0224, -0.0932, -0.0329],
       ...,
       [ 0.0425,  0.0074, -0.0151, ..., -0.0065, -0.0142, -0.0196],
       [-0.0185, -0.0584,  0.0034, ..., -0.0485,  0.0274,  0.044 ],
       [ 0.0532,  0.1203,  0.0299, ...,  0.0844,  0.0441,  0.0336]],
      dtype=float32)

In [24]:
Y_train[:10]

array([[-0.0138, -0.0218, -0.0034, ..., -0.0052,  0.0157, -0.0186],
       [-0.0124, -0.0632,  0.0116, ...,  0.0112,  0.058 , -0.0469],
       [ 0.0069, -0.1661,  0.0196, ..., -0.0125, -0.0673, -0.0716],
       ...,
       [ 0.0626,  0.0079,  0.0121, ...,  0.0295,  0.0339, -0.0395],
       [ 0.3072, -0.2212, -0.4329, ..., -0.0355,  0.0592, -0.0176],
       [ 0.0449,  0.0201, -0.0507, ...,  0.038 , -0.0139, -0.021 ]],
      dtype=float32)

## Embedding space mapping

Let $x_i \in \mathrm{R}^d$ be the distributed representation of word $i$ in the source language, and $y_i \in \mathrm{R}^d$ is the vector representation of its translation. Our purpose is to learn such linear transform $W$ that minimizes euclidian distance between $Wx_i$ and $y_i$ for some subset of word embeddings. Thus we can formulate so-called Procrustes problem:

$$W^*= \arg\min_W \sum_{i=1}^n||Wx_i - y_i||_2$$
or
$$W^*= \arg\min_W ||WX - Y||_F$$

where $||*||_F$ - Frobenius norm.

In Greek mythology, Procrustes or "the stretcher" was a rogue smith and bandit from Attica who attacked people by stretching them or cutting off their legs, so as to force them to fit the size of an iron bed. We make same bad things with source embedding space. Our Procrustean bed is target embedding space.

![embedding_mapping.png](https://github.com/yandexdataschool/nlp_course/raw/master/resources/embedding_mapping.png)

![procrustes.png](https://github.com/yandexdataschool/nlp_course/raw/master/resources/procrustes.png)

But wait...$W^*= \arg\min_W \sum_{i=1}^n||Wx_i - y_i||_2$ looks like simple multiple linear regression (without intercept fit). So let's code.

In [30]:
from sklearn.linear_model import LinearRegression

mapping = LinearRegression()
mapping.fit(X_train, Y_train)

LinearRegression()

In [31]:
y_test = mapping.predict(X_test)

Let's take a look at neigbours of the vector of word _"серпень"_ (_"август"_ in Russian) after linear transform.

In [33]:
august = mapping.predict(uk_emb["серпень"].reshape(1, -1))
ru_emb.most_similar(august)

[('апрель', 0.8541591763496399),
 ('июнь', 0.8411962985992432),
 ('март', 0.8397400379180908),
 ('сентябрь', 0.8359215259552002),
 ('февраль', 0.8328747749328613),
 ('октябрь', 0.8311805725097656),
 ('ноябрь', 0.827814519405365),
 ('июль', 0.8236350417137146),
 ('август', 0.8120611310005188),
 ('декабрь', 0.8037999272346497)]

We can see that neighbourhood of this embedding cosists of different months, but right variant is on the ninth place.

As quality measure we will use precision top-1, top-5 and top-10 (for each transformed Ukrainian embedding we count how many right target pairs are found in top N nearest neighbours in Russian embedding space).

In [36]:
def precision(pairs, mapped_vectors, topn=1):
    """
    :args:
        pairs = list of right word pairs [(uk_word_0, ru_word_0), ...]
        mapped_vectors = list of embeddings after mapping from source embedding space to destination embedding space
        topn = the number of nearest neighbours in destination embedding space to choose from
    :returns:
        precision_val, float number, total number of words for those we can find right translation at top K.
    """
    assert len(pairs) == len(mapped_vectors)

    
    num_matches = 0
    for i, (_, ru) in enumerate(pairs):
        vec = mapped_vectors[i]
        candidates = ru_emb.most_similar(np.asarray(vec).reshape(1, -1), topn=topn)
        top_words = [w for w, _ in candidates]
        if ru in top_words:
            num_matches += 1  
        # YOUR CODE HERE
    precision_val = num_matches / len(pairs)
    return precision_val


In [37]:
assert precision([("серпень", "август")], august, topn=5) == 0.0
assert precision([("серпень", "август")], august, topn=9) == 1.0
assert precision([("серпень", "август")], august, topn=10) == 1.0

In [38]:
assert precision(uk_ru_test, X_test) == 0.0
assert precision(uk_ru_test, Y_test) == 1.0

In [39]:
precision_top1 = precision(uk_ru_test, mapping.predict(X_test), 1)
precision_top5 = precision(uk_ru_test, mapping.predict(X_test), 5)

assert precision_top1 >= 0.635
assert precision_top5 >= 0.8113

## Making it better (orthogonal Procrustean problem)

It can be shown (see original paper) that a self-consistent linear mapping between semantic spaces should be orthogonal. 
We can restrict transform $W$ to be orthogonal. Then we will solve next problem:

$$W^*= \arg\min_W ||WX - Y||_F \text{, where: } W^TW = I$$

$$I \text{- identity matrix}$$

Instead of making yet another regression problem we can find optimal orthogonal transformation using singular value decomposition. It turns out that optimal transformation $W^*$ can be expressed via SVD components:
$$X^TY=U\Sigma V^T\text{, singular value decompostion}$$
$$W^*=UV^T$$

In [42]:
def learn_transform(X_train, Y_train):
    """ 
    :returns: W* : float matrix[emb_dim x emb_dim] as defined in formulae above
    """
    M = X_train.T @ Y_train
    U, _, Vt = np.linalg.svd(M)
    W = U @ Vt
    return W
    # YOU CODE HERE

In [43]:
W = learn_transform(X_train, Y_train)

In [44]:
ru_emb.most_similar([np.matmul(uk_emb["серпень"], W)])

[('апрель', 0.8237909078598022),
 ('сентябрь', 0.8049712181091309),
 ('март', 0.802565336227417),
 ('июнь', 0.8021842241287231),
 ('октябрь', 0.8001735806465149),
 ('ноябрь', 0.7934483289718628),
 ('февраль', 0.7914120554924011),
 ('июль', 0.790810763835907),
 ('август', 0.7891016602516174),
 ('декабрь', 0.7686371803283691)]

In [45]:
assert precision(uk_ru_test, np.matmul(X_test, W)) >= 0.653
assert precision(uk_ru_test, np.matmul(X_test, W), 5) >= 0.824

## UK-RU Translator

Now we are ready to make simple word-based translator: for each word in source language in shared embedding space we find the nearest in target language.


In [48]:
with open("fairy_tale.txt", "r") as inpf:
    uk_sentences = [line.rstrip().lower() for line in inpf]

In [49]:
def translate(sentence):
    """
    :args:
        sentence - sentence in Ukrainian (str)
    :returns:
        translation - sentence in Russian (str)
        
    * find ukrainian embedding for each word in sentence
    * transform ukrainian embedding vector
    * find nearest russian word and replace
    """
    rus_sentence = []
    ukr_words = sentence.split()
    for ukr_word in ukr_words:
        try:
            rus_word = ru_emb.most_similar([np.matmul(uk_emb[ukr_word], W)], topn=1)[0][0]
        except:
            rus_word = ukr_word
        rus_sentence.append(rus_word)   
    return ' '.join(rus_sentence)
    # YOUR CODE HERE

In [50]:
assert translate(".") == "."
assert translate("1 , 3") == "1 , 3"
assert translate("кіт зловив мишу") == "кот поймал мышку"

In [51]:
for sentence in uk_sentences:
    print("src: {}\ndst: {}\n".format(sentence, translate(sentence)))
    break

src: лисичка - сестричка і вовк - панібрат
dst: лисичка – сестричка и волк – панібрат



Not so bad, right? We can easily improve translation using language model and not one but several nearest neighbours in shared embedding space. But next time.

## Would you like to learn more?

### Articles:
* [Exploiting Similarities among Languages for Machine Translation](https://arxiv.org/pdf/1309.4168)  - entry point for multilingual embedding studies by Tomas Mikolov (the author of W2V)
* [Offline bilingual word vectors, orthogonal transformations and the inverted softmax](https://arxiv.org/pdf/1702.03859) - orthogonal transform for unsupervised MT
* [Word Translation Without Parallel Data](https://arxiv.org/pdf/1710.04087)
* [Loss in Translation: Learning Bilingual Word Mapping with a Retrieval Criterion](https://arxiv.org/pdf/1804.07745)
* [Unsupervised Alignment of Embeddings with Wasserstein Procrustes](https://arxiv.org/pdf/1805.11222)

### Repos (with ready-to-use multilingual embeddings):
* https://github.com/facebookresearch/MUSE

* https://github.com/Babylonpartners/fastText_multilingual -